<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 36px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; Technology network &mdash; which fields co-occur</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        Which technology fields appear together on the same patent families &mdash; the co-occurrence network, and the legend that later steps reuse.
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; <span style="color: #be0f05; font-weight: 600;">created by Riccardo Priore</span>
        &nbsp;&middot;&nbsp; Centro PATLIB, AREA Science Park
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 660px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>What this notebook does</strong>
            <br/>Step&nbsp;1 &nbsp;&middot;&nbsp; Load the dataset from notebook 1<br/>Step&nbsp;2 &nbsp;&middot;&nbsp; Retrieve IPC codes and resolve them to technology fields<br/>Step&nbsp;3 &nbsp;&middot;&nbsp; Build the co-occurrence network<br/>Step&nbsp;4 &nbsp;&middot;&nbsp; Export legend + co-occurrence tables
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 660px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;Part 2 of 3 &mdash; the outputs below are already computed.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            Re-running needs <strong>EPO&nbsp;TIP</strong> (it queries PATSTAT&nbsp;PROD via <code>epo.tipdata</code>). Run the three notebooks of this module in order &mdash; each one writes what the next one reads.
        </div>
    </div>
    <div style="margin-top: 20px; font-size: 12px; color: #cbd5e1;">
        Part of EPO TIP Working Group Sessions, 2026. &nbsp;Data: EPO PATSTAT Global.
    </div>
</div>

# Bacterial Antibiotic Resistance IPC Classification Co-occurrence Network Analysis

### Enhanced analysis with technology field mapping and interactive network visualization

**Dataset Source:** Antibiotic Resistance Patent Analysis (Year 2000 onwards)

**Focus:** Bacterial antibiotic resistance ONLY (excluding cancer drug resistance)

**Strategy:** Keywords AND (IPC OR CPC) Classification Codes

## Step 1: Initialize PATSTAT Connection

This cell sets up the Python environment by importing all required libraries and connecting to the PATSTAT production database. It also imports the specific database table models that will be queried throughout the analysis.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import and_, or_, func
from sqlalchemy.orm import aliased
from math import sqrt

# Importing the patstat client
from epo.tipdata.patstat import PatstatClient

# Initialize the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

# Import all required tables
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS209_APPLN_IPC,
    TLS224_APPLN_CPC
)

print("✅ PATSTAT connection initialized")
print("✅ All required tables imported")

✅ PATSTAT connection initialized
✅ All required tables imported


## Step 2: Load Patent Families from Excel Dataset

This cell loads the pre-built antibiotic resistance patent dataset from an Excel file. It extracts the list of unique DOCDB patent family IDs that will be used as the basis for all subsequent database queries and network analysis.

In [2]:
# Load the antibiotic resistance dataset
# Use the most recent Excel file from the main analysis
import glob

# Find the most recent dataset file
dataset_files = glob.glob('1_dataset_and_search_strategy_output/dataset.xlsx')
if dataset_files:
    dataset_file = max(dataset_files)  # Get most recent
    print(f"📂 Loading dataset: {dataset_file}")
    
    df_families = pd.read_excel(dataset_file)
    print(f"✅ Loaded dataset: {len(df_families):,} records")
    
    # Extract unique docdb_family_id values
    if 'docdb_family_id' in df_families.columns:
        all_families = [int(family) for family in df_families['docdb_family_id'].dropna().unique()]
        print(f"📊 Number of distinct patent families: {len(all_families):,}")
    else:
        print("❌ Error: docdb_family_id column not found!")
        print("Available columns:", df_families.columns.tolist())
else:
    print("❌ No dataset files found!")
    print("Please run the main analysis notebook first to generate the dataset.")

📂 Loading dataset: /home/jovyan/training/Patlib Sessions/output_Antibiotic_Resistance_Patent_Analysis_CLEAN/Antibiotic_Resistance_Dataset.xlsx
✅ Loaded dataset: 3,974 records
📊 Number of distinct patent families: 3,974


## Step 3: Define IPC Classification to Technology Field Mapping
### Focused on Antibiotic Resistance Technologies

This cell creates a lookup dictionary that maps IPC classification codes to plain-English technology field names relevant to antibiotic resistance research. It also defines a helper function that resolves any IPC code to its best matching label by trying progressively shorter code prefixes.

In [3]:
# IPC Classification to Technology Field Mapping for Antibiotic Resistance
ipc_to_tech_field = {
    # Pharmaceutical Preparations (A61K)
    'A61K  31': 'Antibiotic Drugs - Organic Compounds',
    'A61K  38': 'Antibiotic Drugs - Peptides',
    'A61K  39': 'Antibiotic Drugs - Antigens/Antibodies',
    'A61K  45': 'Antibiotic Drug Combinations',
    'A61K  47': 'Antibiotic Drug Formulations',
    'A61K   9': 'Antibiotic Drug Delivery',
    
    # Therapeutic Activity (A61P)
    'A61P  31': 'Antiinfective Therapeutics',
    'A61P  43': 'Drug Screening Methods',
    
    # Diagnostics & Testing (C12Q, G01N)
    'C12Q   1': 'Microorganism Detection & Testing',
    'C12Q   3': 'Microbial Activity Testing',
    'G01N  33': 'Diagnostic Analysis Methods',
    'G01N  27': 'Electrochemical Analysis',
    'G01N  21': 'Optical Analysis',
    'G01N  15': 'Particle Analysis',
    
    # Microorganisms (C12N)
    'C12N   1': 'Bacteria & Microorganisms',
    'C12N   5': 'Cell Culture',
    'C12N  15': 'Genetic Engineering',
    'C12N   9': 'Enzymes',
    
    # Antibiotic Production (C12P)
    'C12P   1': 'Antibiotic Fermentation',
    'C12P  17': 'Peptide Production',
    'C12P  21': 'Protein Production',
    
    # Chemistry - Antibiotics (C07)
    'C07D': 'Antibiotic Chemistry - Heterocyclic',
    'C07K': 'Antibiotic Chemistry - Peptides',
    'C07K   7': 'Short Peptide Antibiotics',
    'C07K  14': 'Therapeutic Peptides',
    'C07K  16': 'Antibody Therapeutics',
    'C07C': 'Antibiotic Chemistry - Organic',
    'C07H': 'Antibiotic Chemistry - Carbohydrates',
    
    # Measuring & Testing (G01N)
    'G01N': 'Analytical Testing',
    
    # Medical Devices (A61)
    'A61B': 'Medical Diagnostic Devices',
    'A61L': 'Antimicrobial Materials & Sterilization',
    'A61L  31': 'Antimicrobial Coatings',
    'A61L   2': 'Sterilization Methods',
    'A61M': 'Medical Treatment Devices',
    
    # Inorganic Chemistry (C01)
    'C01B': 'Inorganic Compounds',
    'C01C': 'Metal Compounds',
    'C01G': 'Metal Compounds',
    
    # Biochemistry (C12)
    'C12': 'Biochemistry & Biotechnology',
    
    # Default categories for common prefixes
    'A01': 'Agriculture & Biocides',
    'A23': 'Food Preservation',
    'A61': 'Medical & Pharmaceutical Technology',
    'B01': 'Chemical Processing',
    'B82': 'Nanotechnology',
    'C02': 'Water Treatment',
    'C07': 'Organic Chemistry',
    'C08': 'Polymer Chemistry',
    'C12': 'Biochemistry',
    'G01': 'Measuring & Testing',
    'G06': 'Computing & Data Processing'
}

def get_technology_field(ipc_code):
    """Map IPC code to technology field description for antibiotic resistance"""
    if not ipc_code:
        return "Unknown Technology"
    
    ipc_code = str(ipc_code).strip()
    
    # Try exact match first (8 characters)
    if len(ipc_code) >= 8:
        key = ipc_code[:8]
        if key in ipc_to_tech_field:
            return ipc_to_tech_field[key]
    
    # Try 4-character match
    if len(ipc_code) >= 4:
        prefix = ipc_code[:4]
        if prefix in ipc_to_tech_field:
            return ipc_to_tech_field[prefix]
    
    # Try 3-character prefix match
    if len(ipc_code) >= 3:
        prefix = ipc_code[:3]
        if prefix in ipc_to_tech_field:
            return ipc_to_tech_field[prefix]
    
    # Default fallback
    return f"Technology Field ({ipc_code[:3]})"

print("✅ IPC to Antibiotic Resistance Technology Field mapping created!")
print(f"📊 Total mapping entries: {len(ipc_to_tech_field)}")

✅ IPC to Antibiotic Resistance Technology Field mapping created!
📊 Total mapping entries: 48


## Step 4: Query IPC Classifications and Create Rankings

This cell queries PATSTAT to retrieve all IPC classification codes assigned to the patent families in the dataset. It then counts how many distinct patent families carry each IPC code, attaches a human-readable technology field label, and ranks the codes by frequency.

In [4]:
# Get IPC classifications for all patent families
print("🔍 Querying IPC classifications...\n")

ipc_query = (
    db.query(
        TLS201_APPLN.docdb_family_id.label('patent_family_id'),
        TLS201_APPLN.earliest_filing_year.label('earliest_filing_year'),
        TLS209_APPLN_IPC.ipc_class_symbol.label('IPC')
    )
    .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(all_families))
    .filter(TLS201_APPLN.earliest_filing_year >= 2000)
)

# Execute the query
result = ipc_query.all()

# Convert to DataFrame
df_ipc = pd.DataFrame(result, columns=['patent_family_id', 'earliest_filing_year', 'IPC'])

# Truncate IPC codes to 8 characters for consistency
df_ipc['IPC_8'] = df_ipc['IPC'].astype(str).apply(lambda x: x[:8])

# Count patent families by IPC code
ipc_counts = df_ipc.groupby('IPC_8')['patent_family_id'].nunique().reset_index()
ipc_counts.columns = ['IPC_Code', 'Family_Count']

# Add technology field descriptions
ipc_counts['Technology_Field'] = ipc_counts['IPC_Code'].apply(get_technology_field)

# Sort by family count (descending)
ipc_counts = ipc_counts.sort_values('Family_Count', ascending=False)

print(f"📊 Total IPC codes found: {len(ipc_counts):,}")
print(f"🏠 Total patent families: {df_ipc['patent_family_id'].nunique():,}")
print(f"\n🔝 Top 20 IPC codes by patent family count:")
print(ipc_counts.head(20).to_string(index=False))

🔍 Querying IPC classifications...

📊 Total IPC codes found: 703
🏠 Total patent families: 3,974

🔝 Top 20 IPC codes by patent family count:
IPC_Code  Family_Count                     Technology_Field
A61P  31          1941           Antiinfective Therapeutics
A61K  31          1385 Antibiotic Drugs - Organic Compounds
C12Q   1           960    Microorganism Detection & Testing
C12N  15           768                  Genetic Engineering
C12N   1           570            Bacteria & Microorganisms
C12R   1           553                         Biochemistry
A61K  38           516          Antibiotic Drugs - Peptides
A61K  45           401         Antibiotic Drug Combinations
A61K   9           361             Antibiotic Drug Delivery
A61K  47           335         Antibiotic Drug Formulations
G01N  33           310          Diagnostic Analysis Methods
A61K  35           307  Medical & Pharmaceutical Technology
A61K  36           233  Medical & Pharmaceutical Technology
C07K  14           22

## Step 5: Apply Threshold Filter for Network Analysis

This cell applies a minimum frequency threshold to focus the network on the most relevant IPC codes. Any IPC code appearing in fewer than 50 patent families is dropped, leaving only the codes that represent meaningful technology areas in the dataset.

In [5]:
# Apply threshold for focused network visualization
FAMILY_THRESHOLD = 50  # Threshold for antibiotic resistance network

# Filter IPC codes above threshold
significant_ipc_codes = ipc_counts[ipc_counts['Family_Count'] >= FAMILY_THRESHOLD]
significant_ipc_list = significant_ipc_codes['IPC_Code'].tolist()

print(f"🎯 Threshold: {FAMILY_THRESHOLD} patent families")
print(f"📊 Significant IPC codes above threshold: {len(significant_ipc_list)}")
print(f"🏠 Total families represented: {significant_ipc_codes['Family_Count'].sum():,}")

# Display the significant IPC codes
print(f"\n📋 Significant IPC codes for network analysis:")
print(significant_ipc_codes[['IPC_Code', 'Technology_Field', 'Family_Count']].to_string(index=False))

🎯 Threshold: 50 patent families
📊 Significant IPC codes above threshold: 59
🏠 Total families represented: 13,372

📋 Significant IPC codes for network analysis:
IPC_Code                       Technology_Field  Family_Count
A61P  31             Antiinfective Therapeutics          1941
A61K  31   Antibiotic Drugs - Organic Compounds          1385
C12Q   1      Microorganism Detection & Testing           960
C12N  15                    Genetic Engineering           768
C12N   1              Bacteria & Microorganisms           570
C12R   1                           Biochemistry           553
A61K  38            Antibiotic Drugs - Peptides           516
A61K  45           Antibiotic Drug Combinations           401
A61K   9               Antibiotic Drug Delivery           361
A61K  47           Antibiotic Drug Formulations           335
G01N  33            Diagnostic Analysis Methods           310
A61K  35    Medical & Pharmaceutical Technology           307
A61K  36    Medical & Pharmaceutic

## Step 6: IPC Co-occurrence Analysis

This cell queries PATSTAT for all pairs of IPC codes that appear together on the same patent application, then counts how often each unique pair co-occurs across the dataset. Only pairs where both IPC codes are in the significant list are kept, giving the edge weights for the network.

In [6]:
# Create alias for self-join
TLS209_APPLN_IPC_2 = aliased(TLS209_APPLN_IPC)

print("🔍 Analyzing IPC co-occurrences...\n")

# Get co-occurrence data for significant IPC codes only
co_occurrence_query = (
    db.query(
        TLS201_APPLN.docdb_family_id.label('patent_family_id'),
        TLS201_APPLN.earliest_filing_year.label('earliest_filing_year'),
        TLS209_APPLN_IPC.ipc_class_symbol.label('IPC_1'),
        TLS209_APPLN_IPC_2.ipc_class_symbol.label('IPC_2')
    )
    .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
    .join(TLS209_APPLN_IPC_2, TLS201_APPLN.appln_id == TLS209_APPLN_IPC_2.appln_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(all_families))
    .filter(TLS201_APPLN.earliest_filing_year >= 2000)
    .filter(and_(
        TLS209_APPLN_IPC.ipc_class_symbol > TLS209_APPLN_IPC_2.ipc_class_symbol,
        func.left(TLS209_APPLN_IPC.ipc_class_symbol, 8) != func.left(TLS209_APPLN_IPC_2.ipc_class_symbol, 8)
    ))
)

# Execute query
result = co_occurrence_query.all()

# Convert to DataFrame
df_cooccur = pd.DataFrame(result, columns=['patent_family_id', 'earliest_filing_year', 'IPC_1', 'IPC_2'])

# Truncate IPC codes to 8 characters
df_cooccur['IPC_1_8'] = df_cooccur['IPC_1'].astype(str).apply(lambda x: x[:8])
df_cooccur['IPC_2_8'] = df_cooccur['IPC_2'].astype(str).apply(lambda x: x[:8])

# Filter for significant IPC codes only
df_cooccur_filtered = df_cooccur[
    (df_cooccur['IPC_1_8'].isin(significant_ipc_list)) & 
    (df_cooccur['IPC_2_8'].isin(significant_ipc_list))
].copy()

print(f"📊 Total co-occurrence records: {len(df_cooccur):,}")
print(f"🔍 Filtered co-occurrence records: {len(df_cooccur_filtered):,}")

# Remove duplicates
df_cooccur_unique = df_cooccur_filtered.drop_duplicates()

# Create normalized IPC pairs (sorted order)
df_cooccur_unique['IPC_pair'] = df_cooccur_unique.apply(
    lambda row: tuple(sorted([row['IPC_1_8'], row['IPC_2_8']])), axis=1
)

# Count co-occurrences
cooccur_counts = df_cooccur_unique.groupby('IPC_pair').size().reset_index(name='cooccurrence_count')

# Split pairs back into separate columns
cooccur_counts[['IPC_A', 'IPC_B']] = pd.DataFrame(cooccur_counts['IPC_pair'].tolist(), index=cooccur_counts.index)

# Add technology field descriptions
cooccur_counts['Tech_Field_A'] = cooccur_counts['IPC_A'].apply(get_technology_field)
cooccur_counts['Tech_Field_B'] = cooccur_counts['IPC_B'].apply(get_technology_field)

# Sort by co-occurrence count
cooccur_counts = cooccur_counts.sort_values('cooccurrence_count', ascending=False)

print(f"\n🔗 Unique IPC pairs: {len(cooccur_counts):,}")
print(f"\n🔝 Top 10 IPC co-occurrences:")
print(cooccur_counts[['IPC_A', 'Tech_Field_A', 'IPC_B', 'Tech_Field_B', 'cooccurrence_count']].head(10).to_string(index=False))

🔍 Analyzing IPC co-occurrences...

📊 Total co-occurrence records: 199,422
🔍 Filtered co-occurrence records: 122,210

🔗 Unique IPC pairs: 1,215

🔝 Top 10 IPC co-occurrences:
   IPC_A                           Tech_Field_A    IPC_B                         Tech_Field_B  cooccurrence_count
A61K  31   Antibiotic Drugs - Organic Compounds A61P  31           Antiinfective Therapeutics                4388
A61K  31   Antibiotic Drugs - Organic Compounds A61K  45         Antibiotic Drug Combinations                1718
A61K  31   Antibiotic Drugs - Organic Compounds A61K  47         Antibiotic Drug Formulations                1394
A61K   9               Antibiotic Drug Delivery A61K  31 Antibiotic Drugs - Organic Compounds                1155
A61K  31   Antibiotic Drugs - Organic Compounds A61K  38          Antibiotic Drugs - Peptides                 987
A61K  47           Antibiotic Drug Formulations A61P  31           Antiinfective Therapeutics                 947
A61K   9               Antibi

/tmp/ipykernel_5802/3182745090.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cooccur_unique['IPC_pair'] = df_cooccur_unique.apply(


## Step 7: Create Technology Network Visualization

This cell builds the network graph object from the filtered IPC data. It groups IPC codes that share the same simplified technology label into single nodes, sums their patent family counts, then adds weighted edges between nodes whose IPC codes co-occur in the same patent families.

In [7]:
import networkx as nx
import plotly.graph_objects as go
import plotly.offline as pyo

# Create simplified technology names for better readability
def get_simplified_tech_name(tech_field, ipc_code):
    """Create short, readable technology names"""
    simplifications = {
        'Antibiotic Drugs - Organic Compounds': 'Antibiotic Drugs (Organic)',
        'Antibiotic Drugs - Peptides': 'Antibiotic Drugs (Peptides)',
        'Antibiotic Drugs - Antigens/Antibodies': 'Antibodies & Antigens',
        'Antibiotic Drug Combinations': 'Drug Combinations',
        'Antibiotic Drug Formulations': 'Drug Formulations',
        'Antibiotic Drug Delivery': 'Drug Delivery',
        'Antiinfective Therapeutics': 'Antiinfective Therapy',
        'Microorganism Detection & Testing': 'Microbial Testing',
        'Microbial Activity Testing': 'Activity Testing',
        'Diagnostic Analysis Methods': 'Diagnostic Analysis',
        'Bacteria & Microorganisms': 'Bacteria',
        'Genetic Engineering': 'Genetic Engineering',
        'Antibiotic Chemistry - Heterocyclic': 'Chemistry (Heterocyclic)',
        'Antibiotic Chemistry - Peptides': 'Chemistry (Peptides)',
        'Antibiotic Chemistry - Organic': 'Chemistry (Organic)',
        'Antimicrobial Materials & Sterilization': 'Antimicrobial Materials',
        'Medical & Pharmaceutical Technology': 'Medical Technology',
        'Biochemistry & Biotechnology': 'Biochemistry'
    }
    
    return simplifications.get(tech_field, tech_field)

# PRE-AGGREGATE DATA BY SIMPLIFIED NAME
print("🔧 Aggregating IPC codes that map to same technology field...\n")

# Add simplified name column
significant_ipc_codes_copy = significant_ipc_codes.copy()
significant_ipc_codes_copy['simplified_name'] = significant_ipc_codes_copy.apply(
    lambda row: get_simplified_tech_name(row['Technology_Field'], row['IPC_Code']),
    axis=1
)

# Aggregate by simplified name
aggregated_by_name = significant_ipc_codes_copy.groupby('simplified_name').agg({
    'Family_Count': 'sum',  # SUM the family counts
    'IPC_Code': lambda x: ', '.join(sorted(x)),  # Combine IPC codes
    'Technology_Field': 'first'
}).reset_index()

print(f"📊 After aggregation: {len(aggregated_by_name)} unique technology nodes")
print(f"   (from {len(significant_ipc_codes)} IPC codes)\n")

# Show which nodes represent multiple IPC codes
multi_ipc = aggregated_by_name[aggregated_by_name['IPC_Code'].str.contains(',')]
if len(multi_ipc) > 0:
    print("🔍 Technology fields representing multiple IPC codes:")
    for _, row in multi_ipc.iterrows():
        ipcs = row['IPC_Code'].split(', ')
        print(f"   {row['simplified_name']}: {len(ipcs)} IPC codes ({row['Family_Count']:,} families)")
        for ipc in ipcs:
            count = significant_ipc_codes_copy[significant_ipc_codes_copy['IPC_Code']==ipc]['Family_Count'].values[0]
            print(f"      - {ipc}: {count:,} families")
    print()

# Create network graph with aggregated data
G = nx.Graph()
tech_mapping = {}

# Add nodes from aggregated data
for _, row in aggregated_by_name.iterrows():
    simplified_name = row['simplified_name']
    tech_mapping[simplified_name] = row['IPC_Code']
    
    G.add_node(
        simplified_name,
        ipc_code=row['IPC_Code'],
        technology_field=row['Technology_Field'],
        family_count=row['Family_Count'],
        size=sqrt(row['Family_Count'])
    )

# Add edges (using aggregated nodes)
EDGE_THRESHOLD = 5
significant_cooccur = cooccur_counts[
    (cooccur_counts['IPC_A'].isin(significant_ipc_list)) & 
    (cooccur_counts['IPC_B'].isin(significant_ipc_list)) &
    (cooccur_counts['cooccurrence_count'] >= EDGE_THRESHOLD)
]

# Aggregate edge weights for merged nodes
edge_data = {}
for _, row in significant_cooccur.iterrows():
    tech_a = get_simplified_tech_name(row['Tech_Field_A'], row['IPC_A'])
    tech_b = get_simplified_tech_name(row['Tech_Field_B'], row['IPC_B'])
    
    if tech_a in G.nodes() and tech_b in G.nodes() and tech_a != tech_b:
        edge_key = tuple(sorted([tech_a, tech_b]))
        if edge_key not in edge_data:
            edge_data[edge_key] = 0
        edge_data[edge_key] += row['cooccurrence_count']

# Add aggregated edges to graph
for (tech_a, tech_b), weight in edge_data.items():
    G.add_edge(tech_a, tech_b, weight=weight, cooccurrence_count=weight)

print(f"🕸️ Network created: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

🔧 Aggregating IPC codes that map to same technology field...

📊 After aggregation: 29 unique technology nodes
   (from 59 IPC codes)

🔍 Technology fields representing multiple IPC codes:
   Agriculture & Biocides: 7 IPC codes (709 families)
      - A01N  25: 56 families
      - A01N  37: 75 families
      - A01N  43: 93 families
      - A01N  59: 71 families
      - A01N  63: 151 families
      - A01P   1: 180 families
      - A01P   3: 83 families
   Biochemistry: 3 IPC codes (812 families)
      - C12M   1: 113 families
      - C12N   7: 146 families
      - C12R   1: 553 families
   Chemistry (Heterocyclic): 6 IPC codes (452 families)
      - C07D 401: 64 families
      - C07D 403: 52 families
      - C07D 417: 60 families
      - C07D 471: 124 families
      - C07D 487: 71 families
      - C07D 501: 81 families
   Food Preservation: 4 IPC codes (442 families)
      - A23K  10: 110 families
      - A23K  20: 129 families
      - A23L   3: 82 families
      - A23L  33: 121 families
 

## Step 8: Create Interactive HTML Visualization

This cell builds the interactive Plotly network diagram. It computes a spring layout for node positions, styles each edge by co-occurrence strength, and sizes each node by its patent family count. The finished chart is saved as a standalone HTML file that can be opened in any browser.

In [8]:
# Create enhanced visualization
pos = nx.spring_layout(G, k=3, iterations=50, seed=42)

# Calculate network statistics
degree_centrality = nx.degree_centrality(G)
edge_weights = [G.edges[edge]['weight'] for edge in G.edges()]
avg_weight = np.mean(edge_weights) if edge_weights else 0

# Create edge traces
edge_traces = []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    weight = G.edges[edge]['weight']
    
    # Determine relationship strength
    if weight > avg_weight * 2:
        strength = "Very Strong"
        line_color = 'rgba(50, 50, 50, 0.8)'
    elif weight > avg_weight:
        strength = "Strong" 
        line_color = 'rgba(80, 80, 80, 0.7)'
    else:
        strength = "Moderate"
        line_color = 'rgba(125, 125, 125, 0.5)'
    
    edge_trace = go.Scatter(
        x=[x0, x1, None], y=[y0, y1, None],
        mode='lines',
        line=dict(width=max(2, min(8, weight/10)), color=line_color),
        hoverinfo='text',
        hovertext=f"<b>{edge[0]} ↔ {edge[1]}</b><br>" +
                  f"Co-occurrences: <b>{weight:,}</b><br>" +
                  f"Relationship Strength: <b>{strength}</b>",
        showlegend=False
    )
    edge_traces.append(edge_trace)

# Create node trace
node_x = []
node_y = []
node_text = []
node_size = []
node_color = []
node_hovertext = []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_text.append(node)
    
    # Node attributes
    ipc_code = G.nodes[node]['ipc_code']
    tech_field = G.nodes[node]['technology_field']
    family_count = G.nodes[node]['family_count']
    connections = G.degree(node)
    centrality = degree_centrality[node]
    
    # Node size and color
    node_size.append(max(25, min(100, family_count / 10)))
    node_color.append(family_count)
    
    # Get top connections
    neighbor_info = []
    neighbors = list(G.neighbors(node))
    neighbors_with_weights = [(neighbor, G.edges[node, neighbor]['weight']) for neighbor in neighbors]
    neighbors_with_weights.sort(key=lambda x: x[1], reverse=True)
    
    for neighbor, conn_weight in neighbors_with_weights[:5]:
        neighbor_info.append(f"  → {neighbor}: {conn_weight:,} co-occurrences")
    
    if len(neighbors) > 5:
        neighbor_info.append(f"  ... and {len(neighbors)-5} more connections")
    
    neighbor_text = "<br>".join(neighbor_info) if neighbor_info else "No connections"
    
    hover_text = f"<b>{node}</b><br>" + \
                f"IPC Code: <b>{ipc_code}</b><br>" + \
                f"Full Name: {tech_field}<br>" + \
                f"Patent Families: <b>{family_count:,}</b><br>" + \
                f"Network Connections: <b>{connections}</b><br>" + \
                f"Network Centrality: <b>{centrality:.3f}</b><br><br>" + \
                f"<b>Top Connections:</b><br>{neighbor_text}"
    
    node_hovertext.append(hover_text)

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    text=node_text,
    textposition='middle center',
    textfont=dict(size=10, color='black', family='Arial Bold'),
    hoverinfo='text',
    hovertext=node_hovertext,
    marker=dict(
        size=node_size,
        color=node_color,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(
            title=dict(text="Patent Families<br>(Hover for details)", font=dict(size=14)),
            x=1.02,
            thickness=15
        ),
        line=dict(width=2, color='white'),
        opacity=0.9
    )
)

# Create figure
fig = go.Figure(
    data=edge_traces + [node_trace],
    layout=go.Layout(
        title=dict(
            text='Bacterial Antibiotic Resistance Technology Network<br><sub>Hover over nodes for patent family details and connections</sub>',
            font=dict(size=18, family='Arial Bold')
        ),
        showlegend=False,
        hovermode='closest',
        margin=dict(b=100, l=5, r=120, t=80),
        annotations=[
            dict(
                text=f"<b>📊 Network Guide:</b><br>" +
                     f"<b>Nodes:</b> Size ∝ patent families, Color = family count<br>" +
                     f"<b>Edges:</b> Thickness ∝ co-occurrence strength<br>" +
                     f"<b>Focus:</b> Bacterial antibiotic resistance ONLY<br>" +
                     f"<b>Total Nodes:</b> {G.number_of_nodes()} technologies<br>" +
                     f"<b>Total Edges:</b> {G.number_of_edges()} relationships<br>" +
                     f"<i>💡 Hover over circles to see patent family counts and connections!</i>",
                showarrow=False,
                xref="paper", yref="paper",
                x=0.02, y=-0.02,
                xanchor="left", yanchor="bottom",
                font=dict(size=11),
                bgcolor="rgba(248, 248, 248, 0.9)",
                bordercolor="gray",
                borderwidth=1,
                borderpad=8
            )
        ],
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        plot_bgcolor='white',
        height=700
    )
)

# Save visualization
output_filename = '2_technology_network_output/technology_network.html'
pyo.plot(fig, filename=output_filename)

print(f"✅ Network visualization saved as: {output_filename}")
print(f"💡 Hover over circles to see patent family counts and technology connections!")

✅ Network visualization saved as: /home/jovyan/training/Patlib Sessions/output_Antibiotic_Resistance_Network_Analysis/Antibiotic_Resistance_Technology_Network.html
💡 Hover over circles to see patent family counts and technology connections!


## Step 9: Export Results to Excel Files

This cell exports all analysis results to disk. It saves the interactive network visualization as an HTML file and writes three Excel files covering the technology legend with network metrics, the IPC code rankings, and the full co-occurrence dataset.

In [9]:
# Export results
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# 1. Technology Legend Mapping
legend_data = []
for simplified_name, ipc_code in tech_mapping.items():
    if simplified_name in G.nodes():
        legend_data.append({
            'Network_Label': simplified_name,
            'IPC_Code': ipc_code,
            'Technology_Field': G.nodes[simplified_name]['technology_field'],
            'Patent_Families': G.nodes[simplified_name]['family_count'],
            'Network_Connections': G.degree(simplified_name),
            'Network_Centrality': degree_centrality[simplified_name]
        })

legend_df = pd.DataFrame(legend_data)
legend_df = legend_df.sort_values('Patent_Families', ascending=False)

legend_filename = '2_technology_network_output/technology_legend.xlsx'
legend_df.to_excel(legend_filename, index=False)

# 2. IPC Rankings
ipc_ranking_filename = '2_technology_network_output/ipc_rankings.xlsx'
significant_ipc_codes.to_excel(ipc_ranking_filename, index=False)

# 3. Co-occurrence Data
cooccur_filename = '2_technology_network_output/ipc_cooccurrence.xlsx'
cooccur_counts[['IPC_A', 'Tech_Field_A', 'IPC_B', 'Tech_Field_B', 'cooccurrence_count']].to_excel(
    cooccur_filename, index=False
)

print("📊 Files exported:")
print(f"   1. {output_filename}")
print(f"      → Interactive network visualization (HTML)")
print(f"   2. {legend_filename}")
print(f"      → Technology legend and network metrics (Excel)")
print(f"   3. {ipc_ranking_filename}")
print(f"      → IPC code rankings by patent families (Excel)")
print(f"   4. {cooccur_filename}")
print(f"      → IPC co-occurrence data (Excel)")
print("\n✅ Antibiotic Resistance Network Analysis Complete!")

📊 Files exported:
   1. /home/jovyan/training/Patlib Sessions/output_Antibiotic_Resistance_Network_Analysis/Antibiotic_Resistance_Technology_Network.html
      → Interactive network visualization (HTML)
   2. /home/jovyan/training/Patlib Sessions/output_Antibiotic_Resistance_Network_Analysis/Antibiotic_Resistance_Technology_Legend.xlsx
      → Technology legend and network metrics (Excel)
   3. /home/jovyan/training/Patlib Sessions/output_Antibiotic_Resistance_Network_Analysis/Antibiotic_Resistance_IPC_Rankings.xlsx
      → IPC code rankings by patent families (Excel)
   4. /home/jovyan/training/Patlib Sessions/output_Antibiotic_Resistance_Network_Analysis/Antibiotic_Resistance_IPC_Cooccurrence.xlsx
      → IPC co-occurrence data (Excel)

✅ Antibiotic Resistance Network Analysis Complete!


This cell is a placeholder and contains no code to execute. It represents the end of the notebook workflow.